# Agent recipes

`eshana-mcp` serves the engine to an assistant over the Model Context Protocol (`claude mcp add eshana -- eshana-mcp`).
The same tools can be called in-process, which is how to test a recipe before giving it to an agent: build the
request, read the envelope, and trace any value back to its rule with `explain_derivation`.

In [ ]:
from eshana import mcp

tools = mcp.handle({"jsonrpc": "2.0", "id": 1, "method": "tools/list"})["result"]["tools"]
print([t["name"] for t in tools])

In [ ]:
def call(name, **arguments):
    r = mcp.handle({"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": name, "arguments": arguments}})["result"]
    return r.get("structuredContent") if not r["isError"] else r["content"][0]["text"]

birth = dict(date="1990-05-15", time="14:30", timezone="Asia/Kolkata", latitude=28.6139, longitude=77.209)
d = call("explain_derivation", attribute_id="vedic.moon.nakshatra", **birth)
print(d["attribute"]["value"]["name"], "by", d["attribute"]["rule_version"], "from", [p["id"] for p in d["derived_from"]["positions"]])

A refusal comes back as text with its code, so an agent can ask for what is missing:

In [ ]:
print(call("create_vedic_chart", date="1990-05-15", time="14:30"))

Recipe: give an agent `create_vedic_chart` and `explain_derivation`, and ask it to quote the `rule_version` of every
value it mentions. The envelope's `interpretations` list is always empty: any reading is the agent's own text.